# Clean FNO (290×151, 1 current, seed 0): raw vs FNO-TTT vs FNO + differentiable solver — local Mac (MPS)

Same methods, settings and score outputs as the Colab version (Junbo's `learn.py` TTT, `physics_runs.py` L-BFGS-B, `report.py` metric).
TTT runs on the Apple GPU (MPS); the solver runs on all CPU cores; raw FNO on either.

Keep this notebook next to `ttt_core.py` and `solver_worker.py`. Set the three paths in the first cell. Every step resumes after an interruption.

Needs: `pip install torch numpy scipy numba pandas` (PyTorch ≥ 2.3 recommended for MPS FFTs).

In [1]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'   # any op MPS lacks falls back to CPU instead of failing
HERE = os.path.abspath('.')
REPO = os.path.expanduser('~/diffion')                             # Junbo branch of AzusaHe/diffion (cloned below if missing)
CLEAN_DIR = os.path.expanduser('~/Desktop/Final_P/Synthetic Data')  # clean data: synthetic_batch_10h_batch_01_0000_0199 ... _10
OUT = os.path.join(HERE, 'results')                                # all outputs
DATA = os.path.join(OUT, 'data'); os.makedirs(DATA, exist_ok=True)
import torch
dev = 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device', dev, '| torch', torch.__version__, '| CPU cores', os.cpu_count())

device mps | torch 2.14.1 | CPU cores 10


In [2]:
import subprocess
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'Junbo', 'https://github.com/AzusaHe/diffion', REPO], check=True)
print(subprocess.run(['git', '-C', REPO, 'log', '-1', '--format=%h %ad %s'], capture_output=True, text=True).stdout)

Cloning into '/Users/ZirongHe/diffion'...


f1a94a7 Fri Nov 27 22:34:47 2026 -0800 Publish 300-material, 500-update PINN benchmark



Updating files: 100% (8492/8492), done.


## 0. Load the clean 290×151 data (Junbo's split and selected current)

In [3]:
# Build 290x151, one-current arrays for Junbo's train / val / test split from CLEAN_DIR (cached on Drive)
import glob, csv, json, numpy as np
from concurrent.futures import ThreadPoolExecutor
TM = f'{REPO}/one_current/clean_gaussian/trained_models'
mats = list(csv.DictReader(open(f'{TM}/materials.csv')))
files = {int(os.path.basename(p)[9:15]): p for p in glob.glob(f'{CLEAN_DIR}/*/samples/material_*.npz') + glob.glob(f'{CLEAN_DIR}/*/material_*.npz')}
print(len(files), 'material files found')
with np.load(next(iter(files.values()))) as a:
    print({k: a[k].shape for k in a.files if a[k].ndim})

def one(row):
    mid, j = int(row['material_id']), int(row['current_1_source_index_zero_based'])
    with np.load(files[mid]) as a:
        v = a['valid_time_mask'][j]; av = np.flatnonzero(v)
        av = av[np.rint(np.linspace(0, len(av)-1, min(len(av), 151))).astype(int)]   # Junbo's frame pick (= all valid frames when <= 151)
        raw = a['concentration_nd'][j, av].astype(np.float64)
        nx = raw.shape[-1]
        c = raw.reshape(len(av), 290, nx//290).mean(-1) if nx != 290 else raw      # 1160 native cells -> 290 bins (4 cells each)
        out = np.full((1, 151, 290), np.nan, np.float32); out[0, :len(av)] = c
        m = np.zeros((1, 151), bool); m[0, :len(av)] = True
        t = np.zeros((1, 151)); t[0, :len(av)] = a['tau'][av]
        assert np.isclose(a['iota'][j], float(row['current_1_iota']))
        return mid, out, m, t, np.array([a['iota'][j]]), np.r_[a['D_coefficients_01'], a['t_plus_coefficients_01']]

def build_split(name):
    f = f'{DATA}/{name}290_1cur_clean.npz'
    if os.path.exists(f): return dict(np.load(f))
    rows = sorted([r for r in mats if r['split'] == name], key=lambda r: int(r['split_row_zero_based']))
    missing = [r['material_id'] for r in rows if int(r['material_id']) not in files]
    if missing:
        print(f'WARNING: {len(missing)} of {len(rows)} {name} materials missing from CLEAN_DIR (e.g. {missing[:5]}); continuing without them')
        rows = [r for r in rows if int(r['material_id']) in files]
    with ThreadPoolExecutor(8) as ex: res = list(ex.map(one, rows))
    z = dict(ids=np.array([r[0] for r in res]), c=np.stack([r[1] for r in res]), mask=np.stack([r[2] for r in res]),
             tau=np.stack([r[3] for r in res]), iota=np.stack([r[4] for r in res]), theta=np.stack([r[5] for r in res]))
    np.savez(f, **z); print(name, len(z['ids'])); return z

zt, zv, ztr = build_split('test'), build_split('val'), build_split('train')
ex = np.load(f'{TM}/example_material_000002.npz'); m = ex['mask_290'][0]
if 2 in list(zt['ids']):
    i = list(zt['ids']).index(2)
    assert np.abs(zt['c'][i, 0][zt['mask'][i, 0]] - ex['clean_290'][0][m]).max() < 1e-6, 'binning differs from Junbo'
    print('material 2 matches Junbo exactly')
print('test', len(zt['ids']), 'val', len(zv['ids']), 'train', len(ztr['ids']))

1770 material files found
{'D_coefficients_01': (4,), 't_plus_coefficients_01': (4,), 'rho_coefficients_01': (4,), 'iota': (8,), 'current_density_a_m2': (8,), 'stratum_edges_iota': (9,), 'time_s': (151,), 'tau': (151,), 'x_cells_nd': (1160,), 'x_faces_nd': (1161,), 'concentration_nd': (8, 151, 1160), 'valid_time_mask': (8, 151), 'status': (8,), 'message': (8,), 'termination_time_s': (8,), 'stored_time_count': (8,), 'nfev': (8,), 'njev': (8,), 'nlu': (8,), 'n_steps': (8,), 'n_step_restarts': (8,), 'min_concentration_nd': (8,), 'max_concentration_nd': (8,), 'max_inventory_drift_relative': (8,), 'final_relative_concentration_change': (8,), 'final_relative_flux_nonuniformity': (8,), 'final_relative_rhs_residual': (8,), 'final_right_velocity_nd': (8,), 'final_solvent_residual_linf': (8,), 'depletion_detected': (8,), 'depletion_time_s': (8,), 'depletion_time_lower_s': (8,), 'depletion_time_upper_s': (8,), 'depletion_cell_index': (8,), 'depletion_location_x_nd': (8,), 'depletion_concentration

In [4]:
import sys, json, time, math, numpy as np, torch
sys.path.insert(0, HERE)
from ttt_core import setup, load_fno, build, adapt, errors, junbo_score
cg, learn, inference, alpha_V, bb = setup(REPO)
SEED = 0                          # one seed, one current
RATES, STEPS, N_VAL, CHUNK = (1e-5, 1e-4, 1e-3), 150, 100, 64   # learn.py: RATES, STEPS, va[:100], --chunk 64 (lower CHUNK if memory runs out)
COHORT = 300                      # Junbo's solver cohort: first 300 test materials
# TTT residual scale = RMS of dc/dt over the training split, as learn.prepare()
_ct = np.diff(np.nan_to_num(ztr['c'], nan=0.), axis=2)/np.where(ztr['mask'][:,:,1:] & ztr['mask'][:,:,:-1], np.diff(ztr['tau'], axis=2), 1.)[...,None]
_mid = (ztr['mask'][:,:,1:] & ztr['mask'][:,:,:-1])
CT_SCALE = float(np.sqrt((_ct**2*_mid[...,None]).sum()/_mid.sum()/290)); del _ct; print('CT_SCALE', CT_SCALE)
split = json.load(open(cg/'trained_models'/'split_indices.json'))
def sync():
    if dev == 'mps': torch.mps.synchronize()
def free():
    if dev == 'mps': torch.mps.empty_cache()
assert set(zt['ids']) <= set(split['test']['material_ids']) and set(zv['ids']) <= set(split['validation']['material_ids'])
solver_rows = [i for i, m in enumerate(zt['ids']) if m in set(split['test']['material_ids'][:COHORT])]
print('test', len(zt['ids']), '| val', len(zv['ids']), '| solver cohort', len(solver_rows))
model, rms = load_fno(cg, learn, SEED, dev)
d = build(zt, rms, learn, inference, alpha_V, bb, dev, CT_SCALE)
timing = json.load(open(f'{OUT}/timing.json')) if os.path.exists(f'{OUT}/timing.json') else {}
def save_timing(): json.dump(timing, open(f'{OUT}/timing.json', 'w'), indent=1)

CT_SCALE 1.0496313130356163
test 344 | val 136 | solver cohort 255


In [5]:
# MPS self-test: 2 TTT steps on 2 materials on MPS vs CPU; falls back to CPU if MPS fails or disagrees
if dev == 'mps':
    try:
        sub = {k: v[:2] for k, v in zt.items()}
        res = {}
        for dv_ in ('mps', 'cpu'):
            m_, r_ = load_fno(cg, learn, SEED, dv_)
            dd = build(sub, r_, learn, inference, alpha_V, bb, dv_, CT_SCALE)
            res[dv_] = adapt(m_, dd, torch.arange(2, device=dv_), 1e-4, 2, learn)
        diff = float(np.abs(res['mps'] - res['cpu']).max()); print('MPS vs CPU max |Δθ| after 2 TTT steps:', diff)
        assert diff < 1e-3
    except Exception as e:
        print('MPS check failed -> using CPU:', repr(e)[:200]); dev = 'cpu'
        model, rms = load_fno(cg, learn, SEED, dev); d = build(zt, rms, learn, inference, alpha_V, bb, dev, CT_SCALE)
print('running TTT on', dev)

MPS vs CPU max |Δθ| after 2 TTT steps: 5.960464477539062e-07
running TTT on mps


## 1. Raw FNO

In [6]:
sync(); t0 = time.perf_counter()
with torch.no_grad():
    raw = torch.cat([model(x).sigmoid() for x in d['feat'].split(128)]).cpu().numpy()
sync(); timing['fno_test_s'] = time.perf_counter() - t0; save_timing()
np.savez(f'{OUT}/clean__1__fno.npz', ids=zt['ids'], theta=raw, lo=d['bounds'][:,0], hi=d['bounds'][:,1])
de, te = errors(raw, zt['theta'], d['bounds'][:,0], d['bounds'][:,1])
print(f'raw FNO: D median {np.median(de):.2f} %  mean {de.mean():.2f} %   t+ RMSE median {np.median(te):.4f}')

raw FNO: D median 8.96 %  mean 11.02 %   t+ RMSE median 0.0646


## 2. TTT learning rate on 100 validation materials (`learn.score`)

In [7]:
rate_file = f'{OUT}/ttt_rate.json'
if os.path.exists(rate_file):
    rates = json.load(open(rate_file))
else:
    zv100 = {k: v[:N_VAL] for k, v in zv.items()}
    dv = build(zv100, rms, learn, inference, alpha_V, bb, dev, CT_SCALE)
    idx = torch.arange(len(zv100['ids']), device=dev)
    grid = {}; sync(); t0 = time.perf_counter()
    for r in RATES:
        pred = np.concatenate([adapt(model, dv, b, r, STEPS, learn) for b in idx.split(CHUNK)])
        grid[str(r)] = junbo_score(pred, zv100['theta'], dv['bounds']); print(r, grid[str(r)], flush=True)
    sync()
    rates = dict(lr=float(min(grid, key=grid.get)), score=grid, n_val=len(idx), tune_s=time.perf_counter()-t0)
    json.dump(rates, open(rate_file, 'w'), indent=1); del dv; free()
print('TTT learning rate', rates['lr'])

1e-05 0.026274659710799838
0.0001 0.34448095610388163
0.001 0.9596934420912364
TTT learning rate 1e-05


## 3. FNO-TTT on the test set (resumable per chunk)

In [8]:
final = f'{OUT}/clean__1__fno_ttt.npz'
if not os.path.exists(final):
    idx = torch.arange(len(zt['ids']), device=dev); parts = []
    for j, b in enumerate(idx.split(CHUNK)):
        part = f'{OUT}/ttt_partial_{j:03d}.npz'
        if os.path.exists(part):
            z = np.load(part); parts.append(z['theta']); timing[f'ttt_chunk{j}'] = float(z['s']); continue
        sync(); tb = time.perf_counter()
        p = adapt(model, d, b, rates['lr'], STEPS, learn); sync(); s = time.perf_counter()-tb
        np.savez(part, theta=p, s=s); parts.append(p); timing[f'ttt_chunk{j}'] = s; save_timing()
        print(f'chunk {j}: {len(b)} materials, {s:.1f} s', flush=True)
    pred = np.concatenate(parts)
    np.savez(final, ids=zt['ids'], theta=pred, lo=d['bounds'][:,0], hi=d['bounds'][:,1])
    timing['ttt_test_s'] = sum(v for k, v in timing.items() if k.startswith('ttt_chunk')); save_timing()
    for j in range(len(parts)): os.remove(f'{OUT}/ttt_partial_{j:03d}.npz')
pred = np.load(final)['theta']; de, te = errors(pred, zt['theta'], d['bounds'][:,0], d['bounds'][:,1])
print(f'FNO-TTT: D median {np.median(de):.2f} %  mean {de.mean():.2f} %   t+ RMSE median {np.median(te):.4f}')
del d; free()

chunk 0: 64 materials, 154.3 s
chunk 1: 64 materials, 153.9 s
chunk 2: 64 materials, 154.2 s
chunk 3: 64 materials, 153.7 s
chunk 4: 64 materials, 154.0 s
chunk 5: 24 materials, 56.9 s
FNO-TTT: D median 15.02 %  mean 16.19 %   t+ RMSE median 0.0991


## 4. FNO + differentiable solver (CPU, all cores, resumable per material)

In [9]:
import os; os.environ['DIFFION_REPO'] = REPO
from concurrent.futures import ProcessPoolExecutor
sys.path.insert(0, HERE); import solver_worker
fits = f'{OUT}/solver_fits'; os.makedirs(fits, exist_ok=True)
solver_worker.fit((solver_rows[0], f'{DATA}/test290_1cur_clean.npz', f'{OUT}/clean__1__fno.npz', fits))   # compiles numba once
tasks = [(r, f'{DATA}/test290_1cur_clean.npz', f'{OUT}/clean__1__fno.npz', fits) for r in solver_rows]
t0 = time.perf_counter()
with ProcessPoolExecutor(os.cpu_count()) as ex:
    for k, _ in enumerate(ex.map(solver_worker.fit, tasks)):
        if k % 20 == 0: print(k, len(os.listdir(fits)), 'done', f'{time.perf_counter()-t0:.0f} s', flush=True)
recs = [json.load(open(f'{fits}/{r:03d}.json')) for r in solver_rows]
lo = np.nanmin(zt['c'][solver_rows], axis=(1,2,3)); hi = np.nanmax(zt['c'][solver_rows], axis=(1,2,3))
np.savez(f'{OUT}/clean__1__lbfgs_fno.npz', ids=zt['ids'][solver_rows], theta=np.array([r['theta'] for r in recs]), lo=lo, hi=hi,
         elapsed_s=np.array([r['elapsed_s'] for r in recs]), evals=np.array([r['evals'] for r in recs]),
         concentration_rmse=np.array([r['c_rmse'] for r in recs]))
print('statuses:', {s: sum(r['status'] == s for r in recs) for s in {r['status'] for r in recs}})

0 1 done 1 s
20 27 done 52 s
40 50 done 90 s
60 72 done 128 s
80 93 done 161 s
100 111 done 187 s
120 129 done 231 s
140 154 done 268 s
160 180 done 303 s
180 188 done 312 s
200 219 done 360 s
220 225 done 373 s
240 250 done 421 s
statuses: {'STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT': 51, 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH': 188, 'ABNORMAL: ': 11, 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL': 5}


## 5. Scores (Junbo's `report.py` metric) → `results/accuracy_cost_fno_refinement.csv`

In [10]:
import pandas as pd
truth = {m: t for m, t in zip(zt['ids'], zt['theta'])}
runs = {'fno': 'clean__1__fno', 'fno_ttt': 'clean__1__fno_ttt', 'lbfgs_fno': 'clean__1__lbfgs_fno'}
Z = {k: np.load(f'{OUT}/{v}.npz') for k, v in runs.items()}
n_test = len(zt['ids'])
cost = {'fno': timing['fno_test_s']/n_test,
        'fno_ttt': timing['fno_test_s']/n_test + timing['ttt_test_s']/n_test,
        'lbfgs_fno': timing['fno_test_s']/n_test + float(Z['lbfgs_fno']['elapsed_s'].mean())}
common = set(Z['lbfgs_fno']['ids'])
rows = []
for cohort in ('own', 'common'):
    for k, z in Z.items():
        keep = np.array([cohort == 'own' or m in common for m in z['ids']])
        th = np.array([truth[m] for m in z['ids'][keep]])
        de, te = errors(z['theta'][keep], th, z['lo'][keep], z['hi'][keep])
        rows.append(dict(domain='clean', currents=1, method=k, cohort=cohort, n_test=int(keep.sum()),
                         D_median_pct=np.median(de), t_median_rmse=np.median(te), D_mean_pct=de.mean(), t_mean_rmse=te.mean(),
                         median_observed_c_span=float(np.median(z['hi'][keep]-z['lo'][keep])),
                         validation_search_s=rates['tune_s'] if k == 'fno_ttt' else 0.0,
                         pipeline_mean_s_per_material=cost[k]))
df = pd.DataFrame(rows); df.to_csv(f'{OUT}/accuracy_cost_fno_refinement.csv', index=False)
print(df.round(4).to_string(index=False))

domain  currents    method cohort  n_test  D_median_pct  t_median_rmse  D_mean_pct  t_mean_rmse  median_observed_c_span  validation_search_s  pipeline_mean_s_per_material
 clean         1       fno    own     344        8.9609         0.0646     11.0175       0.0752                  1.1217               0.0000                        0.0151
 clean         1   fno_ttt    own     344       15.0154         0.0991     16.1910       0.1131                  1.1217             716.3083                        2.4190
 clean         1 lbfgs_fno    own     255        0.2115         0.0014      2.0876       0.0133                  1.2481               0.0000                       16.5732
 clean         1       fno common     255        9.4986         0.0666     11.2178       0.0774                  1.2481               0.0000                        0.0151
 clean         1   fno_ttt common     255       15.2859         0.1118     16.7744       0.1209                  1.2481             716.3083     